# 04 — Speaker Encoder Fine-Tuning (ECAPA-TDNN)
## TeluguVoiceBridge v2 — Constrained Hardware Plan

**Model:** `speechbrain/spkrec-ecapa-voxceleb` (pretrained on VoxCeleb2)  
**Fine-tuning:** Full model on CPU — only 80 MB, frees GPU for Whisper  
**Training data:** AI4Bharat Rasa Telugu / RAVDESS  
**Loss:** AAM-Softmax (margin=0.2, scale=30)  
**Target:** EER ≤ 12%, cosine similarity same-speaker ≥ 0.70

### Why CPU?
- Model is only ~80 MB — CPU training is feasible (~4-6 hours/epoch)
- Frees GPU for Whisper ASR training (Phases 2 & 3 run in parallel)
- Output: 192-dim L2-normalized speaker embedding

---
## 4.1 — Setup & Config

In [1]:
import os, gc, pathlib, time, json, csv, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
from omegaconf import OmegaConf
from collections import defaultdict

BASE = pathlib.Path(os.getcwd())
CONFIG = OmegaConf.load(BASE / "configs" / "speaker.yaml")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(CONFIG.training.device)  # "cpu"
print(f"Device: {DEVICE}")
print(f"Config: {OmegaConf.to_yaml(CONFIG)}")

Device: cpu
Config: model:
  name: speechbrain/spkrec-ecapa-voxceleb
  embedding_dim: 192
  device: cpu
training:
  device: cpu
  speakers_per_batch: 4
  utterances_per_speaker: 4
  learning_rate: 5.0e-05
  epochs: 20
  weight_decay: 0.0001
  early_stopping_patience: 5
loss:
  type: aam_softmax
  margin: 0.2
  scale: 30
target_metrics:
  eer: 0.12
  cosine_same_speaker: 0.7
  inference_time_sec: 0.5



---
## 4.2 — Load Pretrained ECAPA-TDNN

In [2]:
from speechbrain.inference.speaker import EncoderClassifier

PRETRAINED_DIR = BASE / "checkpoints" / "speaker_encoder" / "pretrained"

print("Loading pretrained ECAPA-TDNN...")
encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir=str(PRETRAINED_DIR),
    run_opts={"device": str(DEVICE)},
)

# Get the actual model modules
model = encoder.mods  # SpeechBrain module dict

param_count = sum(p.numel() for p in encoder.mods.parameters())
param_size_mb = sum(p.numel() * p.element_size() for p in encoder.mods.parameters()) / 1e6
print(f"Parameters: {param_count:,} ({param_size_mb:.1f} MB)")
print(f"✓ ECAPA-TDNN loaded on {DEVICE}.")

Loading pretrained ECAPA-TDNN...


/home/nibiru/.conda/envs/ml_env/lib/python3.11/site-packages/speechbrain/utils/autocast.py:188: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  wrapped_fwd = torch.cuda.amp.custom_fwd(fwd, cast_inputs=cast_inputs)


Parameters: 22,150,912 (88.6 MB)
✓ ECAPA-TDNN loaded on cpu.


---
## 4.3 — AAM-Softmax Loss

In [3]:
class AAMSoftmax(nn.Module):
    """
    Additive Angular Margin Softmax for speaker verification.
    Maps embeddings → speaker classes with angular margin penalty.
    """
    def __init__(self, embedding_dim, num_speakers, margin=0.2, scale=30.0):
        super().__init__()
        self.weight = nn.Parameter(torch.FloatTensor(num_speakers, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        self.margin = margin
        self.scale = scale
        self.cos_m = np.cos(margin)
        self.sin_m = np.sin(margin)
        self.threshold = np.cos(np.pi - margin)
        self.mm = np.sin(np.pi - margin) * margin
        self.ce = nn.CrossEntropyLoss()
    
    def forward(self, embeddings, labels):
        """
        Args:
            embeddings: [batch, embedding_dim] L2-normalized
            labels: [batch] speaker class indices
        Returns:
            loss: scalar
        """
        # L2 normalize
        embeddings = F.normalize(embeddings, dim=1)
        weight = F.normalize(self.weight, dim=1)
        
        # Cosine similarity
        cosine = F.linear(embeddings, weight)  # [batch, num_speakers]
        sine = torch.sqrt((1.0 - cosine.pow(2)).clamp(0, 1))
        
        # cos(θ + m) = cos(θ)cos(m) - sin(θ)sin(m)
        phi = cosine * self.cos_m - sine * self.sin_m
        
        # Numerical stability
        phi = torch.where(cosine > self.threshold, phi, cosine - self.mm)
        
        # One-hot for target class
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.unsqueeze(1).long(), 1)
        
        # Apply margin only to target class
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.scale
        
        return self.ce(output, labels)

print("✓ AAM-Softmax loss defined.")

✓ AAM-Softmax loss defined.


---
## 4.3b — Gradient Reversal Layer & Language Classifier (for language-agnostic embeddings)

The **Gradient Reversal Layer (GRL)** ensures the speaker encoder learns embeddings that do **not** encode language identity.  
- During forward pass: identity transform (embedding passes through unchanged).
- During backward pass: gradients are **negated** (multiplied by −λ).

This forces the encoder to "forget" language, producing embeddings usable across Telugu and English.  
λ ramps from 0 → 1 over the first half of training (annealing schedule from [Ganin et al. 2016]).

In [ ]:
# ═══════════════════════════════════════════════════
# Gradient Reversal Layer (GRL)
# ═══════════════════════════════════════════════════

from torch.autograd import Function

class GradientReversalFunction(Function):
    """Reverses gradients during backward pass (Ganin et al., 2016)."""
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grad_reverse(x, lambd=1.0):
    return GradientReversalFunction.apply(x, lambd)


class LanguageClassifier(nn.Module):
    """
    Adversarial head that tries to predict language from speaker embedding.
    GRL ensures encoder learns to REMOVE language info.
    """
    def __init__(self, embedding_dim=192, hidden_dim=64, num_languages=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, num_languages),
        )
    
    def forward(self, embeddings, lambd=1.0):
        """
        Args:
            embeddings: [batch, 192] speaker embeddings
            lambd: GRL strength (0→1 over training)
        Returns:
            logits: [batch, num_languages]
        """
        reversed_emb = grad_reverse(embeddings, lambd)
        return self.net(reversed_emb)

language_classifier = LanguageClassifier(
    embedding_dim=CONFIG.model.embedding_dim,  # 192
    hidden_dim=64,
    num_languages=2,  # 0=English (RAVDESS), 1=Telugu (FLEURS)
).to(DEVICE)

lang_criterion = nn.CrossEntropyLoss()

print(f"✓ GRL + LanguageClassifier defined.")
print(f"  Language classes: en=0, te=1")
print(f"  Params: {sum(p.numel() for p in language_classifier.parameters()):,}")

---
## 4.4 — Dataset & Contrastive Batch Sampler

In [4]:
from torch.utils.data import Dataset, DataLoader, Sampler

class SpeakerDataset(Dataset):
    """Speaker verification dataset with per-speaker clip grouping."""
    
    def __init__(self, manifest_path, base_dir, split="train", sr=16000, max_dur=8.0):
        df = pd.read_csv(manifest_path)
        self.df = df[df["split"] == split].reset_index(drop=True)
        self.base_dir = pathlib.Path(base_dir)
        self.sr = sr
        self.max_samples = int(max_dur * sr)
        
        # Build speaker → indices mapping
        self.speaker_ids = sorted(self.df["speaker_id"].unique())
        self.speaker_to_label = {s: i for i, s in enumerate(self.speaker_ids)}
        self.speaker_to_indices = defaultdict(list)
        for idx, row in self.df.iterrows():
            self.speaker_to_indices[row["speaker_id"]].append(idx)
        
        print(f"  {split}: {len(self.df)} clips, {len(self.speaker_ids)} speakers")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = self.base_dir / row["audio_path"]
        
        wav, sr = sf.read(str(audio_path))
        if sr != self.sr:
            wav = torchaudio.functional.resample(
                torch.from_numpy(wav).float().unsqueeze(0), sr, self.sr
            ).squeeze(0).numpy()
        
        # Truncate or pad to fixed length
        if len(wav) > self.max_samples:
            # Random crop
            start = random.randint(0, len(wav) - self.max_samples)
            wav = wav[start:start + self.max_samples]
        elif len(wav) < self.max_samples:
            # Pad with silence
            wav = np.pad(wav, (0, self.max_samples - len(wav)))
        
        label = self.speaker_to_label[row["speaker_id"]]
        return torch.from_numpy(wav).float(), torch.tensor(label, dtype=torch.long)


class ContrastiveBatchSampler(Sampler):
    """
    Samples batches with N speakers × M utterances per speaker.
    Ensures diverse speaker representation in each batch.
    """
    def __init__(self, dataset, speakers_per_batch=4, utterances_per_speaker=4):
        self.speaker_to_indices = dataset.speaker_to_indices
        self.speakers = list(self.speaker_to_indices.keys())
        self.spk_per_batch = speakers_per_batch
        self.utt_per_spk = utterances_per_speaker
        self.batch_size = speakers_per_batch * utterances_per_speaker
        
        # Filter speakers with enough utterances
        self.speakers = [s for s in self.speakers 
                        if len(self.speaker_to_indices[s]) >= utterances_per_speaker]
    
    def __iter__(self):
        random.shuffle(self.speakers)
        
        for i in range(0, len(self.speakers) - self.spk_per_batch + 1, self.spk_per_batch):
            batch_speakers = self.speakers[i:i + self.spk_per_batch]
            batch_indices = []
            
            for spk in batch_speakers:
                indices = self.speaker_to_indices[spk]
                selected = random.sample(indices, min(self.utt_per_spk, len(indices)))
                batch_indices.extend(selected)
            
            yield batch_indices
    
    def __len__(self):
        return len(self.speakers) // self.spk_per_batch

print("✓ Dataset and ContrastiveBatchSampler defined.")

✓ Dataset and ContrastiveBatchSampler defined.


---
## 4.4b — Language-Adversarial Dataset (RAVDESS + FLEURS Telugu)

For the GRL branch, we need clips from **both** languages.  
- **RAVDESS** (English) — already in speaker manifest  
- **FLEURS Telugu** — 2303 clips from `../data/raw/fleurs_te_full/train/`  

No speaker labels needed for this dataset — only language labels (en=0, te=1).

In [ ]:
import glob

class LanguageDataset(Dataset):
    """
    Mixed-language dataset for GRL adversarial training.
    Loads English (RAVDESS) and Telugu (FLEURS) clips with language labels.
    """
    def __init__(self, manifest_path, base_dir, fleurs_dir, split="train",
                 sr=16000, max_dur=8.0, balance=True):
        self.sr = sr
        self.max_samples = int(max_dur * sr)
        self.base_dir = pathlib.Path(base_dir)
        
        # ── English clips from speaker manifest (RAVDESS) ──
        df = pd.read_csv(manifest_path)
        df_split = df[df["split"] == split].reset_index(drop=True)
        
        self.english_paths = [
            str(self.base_dir / row["audio_path"]) for _, row in df_split.iterrows()
        ]
        
        # ── Telugu clips from FLEURS ──
        fleurs_path = pathlib.Path(fleurs_dir)
        self.telugu_paths = sorted(glob.glob(str(fleurs_path / "train" / "*.wav")))
        
        if not self.telugu_paths:
            print("  ⚠ No FLEURS Telugu files found — GRL will be English-only")
        
        # Balance: subsample the larger set to match the smaller
        if balance and self.telugu_paths:
            min_count = min(len(self.english_paths), len(self.telugu_paths))
            self.english_paths = random.sample(self.english_paths,
                                               min(min_count, len(self.english_paths)))
            self.telugu_paths = random.sample(self.telugu_paths,
                                              min(min_count, len(self.telugu_paths)))
        
        # Combined list: (path, lang_id)
        self.samples = [(p, 0) for p in self.english_paths] + \
                       [(p, 1) for p in self.telugu_paths]
        random.shuffle(self.samples)
        
        n_en = len(self.english_paths)
        n_te = len(self.telugu_paths)
        print(f"  LanguageDataset: {n_en} English + {n_te} Telugu = {len(self.samples)} total")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, lang_id = self.samples[idx]
        
        try:
            wav, sr = sf.read(path)
        except Exception:
            # Fallback: return silence
            wav = np.zeros(self.max_samples)
            sr = self.sr
        
        if wav.ndim > 1:
            wav = wav[:, 0]  # mono
        
        if sr != self.sr:
            wav = torchaudio.functional.resample(
                torch.from_numpy(wav).float().unsqueeze(0), sr, self.sr
            ).squeeze(0).numpy()
        
        # Truncate or pad
        if len(wav) > self.max_samples:
            start = random.randint(0, len(wav) - self.max_samples)
            wav = wav[start:start + self.max_samples]
        elif len(wav) < self.max_samples:
            wav = np.pad(wav, (0, self.max_samples - len(wav)))
        
        return (torch.from_numpy(wav).float(),
                torch.tensor(lang_id, dtype=torch.long))


# Create language dataset and loader
FLEURS_TE_DIR = BASE.parent / "data" / "raw" / "fleurs_te_full"

lang_dataset = LanguageDataset(
    manifest_path=SPK_MANIFEST,
    base_dir=BASE,
    fleurs_dir=str(FLEURS_TE_DIR),
    split="train",
    sr=16000,
    max_dur=8.0,
    balance=True,
)

lang_loader = DataLoader(
    lang_dataset,
    batch_size=CONFIG.training.speakers_per_batch * CONFIG.training.utterances_per_speaker,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

print(f"Language batches per epoch: {len(lang_loader)}")

In [5]:
# Create datasets and dataloaders
SPK_MANIFEST = BASE / "data" / "metadata" / "speaker_manifest.csv"

if not SPK_MANIFEST.exists():
    print("⚠ Speaker manifest not found. Run notebook 02 first.")
    print("  Or copy from existing pipeline:")
    existing = BASE.parent / "data" / "metadata" / "speaker_manifest.csv"
    if existing.exists():
        import shutil
        shutil.copy2(existing, SPK_MANIFEST)
        print(f"  ✓ Copied from {existing}")

print("Loading speaker datasets...")
train_dataset = SpeakerDataset(SPK_MANIFEST, BASE, split="train")
val_dataset = SpeakerDataset(SPK_MANIFEST, BASE, split="val")

train_sampler = ContrastiveBatchSampler(
    train_dataset,
    speakers_per_batch=CONFIG.training.speakers_per_batch,
    utterances_per_speaker=CONFIG.training.utterances_per_speaker,
)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=train_sampler,
    num_workers=0,  # CPU training — keep workers low
    pin_memory=False,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG.training.speakers_per_batch * CONFIG.training.utterances_per_speaker,
    shuffle=False,
    num_workers=0,
)

num_speakers = len(train_dataset.speaker_ids)
print(f"\nNum speakers (train): {num_speakers}")
print(f"Train batches:        {len(train_sampler)}")

Loading speaker datasets...
  train: 1135 clips, 19 speakers
  val: 120 clips, 2 speakers

Num speakers (train): 19
Train batches:        4


---
## 4.5 — Extract Embeddings Helper

In [6]:
def extract_embedding(encoder, wavs):
    """
    Extract 192-dim speaker embeddings from waveforms.
    
    Args:
        encoder: SpeechBrain EncoderClassifier
        wavs: tensor [batch, time] at 16kHz
    Returns:
        embeddings: tensor [batch, 192] L2-normalized
    """
    with torch.no_grad():
        embeddings = encoder.encode_batch(wavs)
        embeddings = embeddings.squeeze(1)  # [batch, 192]
        embeddings = F.normalize(embeddings, dim=1)  # L2 normalize
    return embeddings

# Quick test
test_wav = torch.randn(2, 16000 * 3)  # 2 clips of 3 seconds
test_emb = extract_embedding(encoder, test_wav)
print(f"Embedding shape: {test_emb.shape}")
print(f"Embedding norm:  {torch.norm(test_emb, dim=1)}")
del test_wav, test_emb
print("✓ Embedding extraction verified.")

Embedding shape: torch.Size([2, 192])
Embedding norm:  tensor([1.0000, 1.0000])
✓ Embedding extraction verified.


---
## 4.6 — Training Setup

In [ ]:
# Loss function
embedding_dim = CONFIG.model.embedding_dim  # 192
criterion = AAMSoftmax(
    embedding_dim=embedding_dim,
    num_speakers=num_speakers,
    margin=CONFIG.loss.margin,
    scale=CONFIG.loss.scale,
).to(DEVICE)

# Optimizer — train encoder, AAMSoftmax, and LanguageClassifier
all_params = (list(encoder.mods.parameters())
              + list(criterion.parameters())
              + list(language_classifier.parameters()))
optimizer = torch.optim.AdamW(
    all_params,
    lr=CONFIG.training.learning_rate,
    weight_decay=CONFIG.training.weight_decay,
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

EPOCHS = CONFIG.training.epochs
PATIENCE = CONFIG.training.early_stopping_patience
CKPT_DIR = BASE / "checkpoints" / "speaker_encoder"
LOG_FILE = BASE / "logs" / "speaker_training_log.csv"

with open(LOG_FILE, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "lang_loss", "val_loss",
                     "val_eer", "grl_lambda", "lr", "time_sec"])

# GRL annealing config
GRL_RAMP_EPOCHS = max(EPOCHS // 2, 1)  # ramp over first half
GRL_WEIGHT = 0.3                        # weight of language loss

print(f"Epochs:       {EPOCHS}")
print(f"Early stop:   patience={PATIENCE}")
print(f"LR:           {CONFIG.training.learning_rate}")
print(f"Loss:         AAM-Softmax (m={CONFIG.loss.margin}, s={CONFIG.loss.scale})")
print(f"GRL:          lambda ramps 0->1 over first {GRL_RAMP_EPOCHS} epochs, weight={GRL_WEIGHT}")
print(f"Language cls: 2 classes (en=0, te=1)")

Epochs:       20
Early stop:   patience=5
LR:           5e-05
Loss:         AAM-Softmax (m=0.2, s=30)


---
## 4.7 — EER Evaluation

In [9]:
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from sklearn.metrics import roc_curve

def compute_eer(encoder, val_loader, device, max_pairs=500):
    """
    Compute Equal Error Rate on validation set.
    Creates same-speaker and different-speaker pairs, computes cosine similarity.
    """
    encoder.mods.eval()
    all_embeddings = []
    all_labels = []
    
    with torch.no_grad():
        for wavs, labels in val_loader:
            wavs = wavs.to(device)
            embs = extract_embedding(encoder, wavs).cpu()
            all_embeddings.append(embs)
            all_labels.append(labels)
    
    all_embeddings = torch.cat(all_embeddings, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    
    # Generate pairs
    n = len(all_embeddings)
    scores = []
    labels = []
    
    pairs_created = 0
    for i in range(n):
        for j in range(i + 1, n):
            if pairs_created >= max_pairs:
                break
            cos_sim = F.cosine_similarity(
                all_embeddings[i:i+1], all_embeddings[j:j+1]
            ).item()
            scores.append(cos_sim)
            labels.append(1 if all_labels[i] == all_labels[j] else 0)
            pairs_created += 1
        if pairs_created >= max_pairs:
            break
    
    if not scores or len(set(labels)) < 2:
        return 0.5, 0.0  # Default if not enough data
    
    # Compute EER
    fpr, tpr, thresholds = roc_curve(labels, scores)
    try:
        eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    except ValueError:
        eer = 0.5
    
    # Average cosine similarity for same-speaker pairs
    same_speaker_scores = [s for s, l in zip(scores, labels) if l == 1]
    avg_same = np.mean(same_speaker_scores) if same_speaker_scores else 0.0
    
    encoder.mods.train()
    return eer, avg_same

print("✓ EER evaluation function ready.")

✓ EER evaluation function ready.


---
## 4.8 — Training Loop (CPU)

In [ ]:
# ═══════════════════════════════════════════════════
# MAIN TRAINING LOOP — Speaker Encoder + GRL
# ═══════════════════════════════════════════════════

best_eer = float("inf")
patience_counter = 0
t_start = time.time()

print("="*60)
print("Starting Speaker Encoder Training with GRL")
print("="*60)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    encoder.mods.train()
    language_classifier.train()
    epoch_spk_loss = 0.0
    epoch_lang_loss = 0.0
    n_batches = 0
    
    # GRL lambda annealing: 0 -> 1 over first half of training
    progress = min(epoch / max(GRL_RAMP_EPOCHS, 1), 1.0)
    grl_lambda = float(2.0 / (1.0 + np.exp(-10.0 * progress)) - 1.0)
    
    # Create language loader iterator (cycles if shorter)
    lang_iter = iter(lang_loader)
    
    for batch_idx, (wavs, labels) in enumerate(train_loader):
        wavs = wavs.to(DEVICE)
        labels = labels.to(DEVICE)
        
        # ── Speaker branch: AAMSoftmax ──
        embeddings = encoder.encode_batch(wavs).squeeze(1)
        embeddings = F.normalize(embeddings, dim=1)
        spk_loss = criterion(embeddings, labels)
        
        # ── Language branch: GRL adversarial ──
        try:
            lang_wavs, lang_labels = next(lang_iter)
        except StopIteration:
            lang_iter = iter(lang_loader)
            lang_wavs, lang_labels = next(lang_iter)
        
        lang_wavs = lang_wavs.to(DEVICE)
        lang_labels = lang_labels.to(DEVICE)
        
        lang_embeddings = encoder.encode_batch(lang_wavs).squeeze(1)
        lang_embeddings = F.normalize(lang_embeddings, dim=1)
        
        # GRL reverses gradients inside LanguageClassifier
        lang_logits = language_classifier(lang_embeddings, lambd=grl_lambda)
        lang_loss = lang_criterion(lang_logits, lang_labels)
        
        # ── Combined loss ──
        total_loss = spk_loss + GRL_WEIGHT * lang_loss
        
        # Backward
        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(all_params, 5.0)
        optimizer.step()
        
        epoch_spk_loss += spk_loss.item()
        epoch_lang_loss += lang_loss.item()
        n_batches += 1
        
        if (batch_idx + 1) % 10 == 0:
            lang_acc = (lang_logits.argmax(1) == lang_labels).float().mean().item()
            print(f"  Epoch {epoch} | Batch {batch_idx+1}/{len(train_sampler)} | "
                  f"SpkL: {spk_loss.item():.3f} LangL: {lang_loss.item():.3f} "
                  f"LangAcc: {lang_acc:.2f} lam: {grl_lambda:.2f}", end="\r")
    
    avg_spk_loss = epoch_spk_loss / max(n_batches, 1)
    avg_lang_loss = epoch_lang_loss / max(n_batches, 1)
    
    # ─── Validation ───
    eer, avg_cosine = compute_eer(encoder, val_loader, DEVICE)
    
    epoch_time = time.time() - epoch_start
    lr = optimizer.param_groups[0]["lr"]
    
    print(f"Epoch {epoch:>2d}/{EPOCHS} | SpkL: {avg_spk_loss:.4f} | "
          f"LangL: {avg_lang_loss:.4f} | EER: {eer:.4f} | "
          f"CosSim: {avg_cosine:.4f} | GRL_lam: {grl_lambda:.3f} | "
          f"LR: {lr:.2e} | {epoch_time/60:.1f}min")
    
    # Log
    with open(LOG_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch, f"{avg_spk_loss:.4f}", f"{avg_lang_loss:.4f}",
                        "", f"{eer:.4f}", f"{grl_lambda:.3f}",
                        f"{lr:.2e}", f"{epoch_time:.0f}"])
    
    # LR scheduling
    scheduler.step(eer)
    
    # ─── Save best ───
    if eer < best_eer:
        best_eer = eer
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": encoder.mods.state_dict(),
            "language_classifier_state_dict": language_classifier.state_dict(),
            "eer": eer,
            "cosine_sim": avg_cosine,
            "grl_lambda": grl_lambda,
        }, CKPT_DIR / "best_model.ckpt")
        print(f"  * New best EER: {eer:.4f} -> saved")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\n  Early stopping at epoch {epoch} "
                  f"(no improvement for {PATIENCE} epochs)")
            break
    
    # Save last model
    torch.save({
        "epoch": epoch,
        "model_state_dict": encoder.mods.state_dict(),
        "language_classifier_state_dict": language_classifier.state_dict(),
        "eer": eer,
    }, CKPT_DIR / "last_model.ckpt")
    
    gc.collect()

total_time = time.time() - t_start
print(f"\n{'='*60}")
print(f"Training complete!")
print(f"Best EER: {best_eer:.4f} (target: <= {CONFIG.target_metrics.eer})")
print(f"Total time: {total_time/3600:.1f} hours")
print(f"{'='*60}")

Starting Speaker Encoder Training (CPU)
Epoch  1/20 | Loss: 10.6251 | EER: 0.0115 | CosSim: 0.5614 | LR: 5.00e-05 | 1.0min
  ★ New best EER: 0.0115 → saved
Epoch  2/20 | Loss: 11.2483 | EER: 0.0154 | CosSim: 0.5130 | LR: 5.00e-05 | 1.0min
Epoch  3/20 | Loss: 11.0084 | EER: 0.0500 | CosSim: 0.4535 | LR: 5.00e-05 | 1.0min
Epoch  4/20 | Loss: 10.5734 | EER: 0.0769 | CosSim: 0.4117 | LR: 5.00e-05 | 1.0min
Epoch  5/20 | Loss: 10.4435 | EER: 0.0923 | CosSim: 0.3831 | LR: 5.00e-05 | 1.0min
Epoch  6/20 | Loss: 10.5217 | EER: 0.1038 | CosSim: 0.3638 | LR: 2.50e-05 | 1.0min

  Early stopping at epoch 6 (no improvement for 5 epochs)

Training complete!
Best EER: 0.0115 (target: ≤ 0.12)
Total time: 0.1 hours


---
## 4.9 — Final Evaluation

In [12]:
# Load best model
best_ckpt = torch.load(CKPT_DIR / "best_model.ckpt", map_location=DEVICE, weights_only=False)
encoder.mods.load_state_dict(best_ckpt["model_state_dict"])
print(f"Loaded best model from epoch {best_ckpt['epoch']}")

# Test set evaluation
test_dataset = SpeakerDataset(SPK_MANIFEST, BASE, split="test")
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

test_eer, test_cosine = compute_eer(encoder, test_loader, DEVICE, max_pairs=1000)

print(f"\n{'='*40}")
print(f"FINAL TEST RESULTS")
print(f"{'='*40}")
print(f"EER:            {test_eer:.4f} (target: ≤ {CONFIG.target_metrics.eer})")
print(f"Cosine (same):  {test_cosine:.4f} (target: ≥ {CONFIG.target_metrics.cosine_same_speaker})")

eer_pass = test_eer <= CONFIG.target_metrics.eer
cos_pass = test_cosine >= CONFIG.target_metrics.cosine_same_speaker
print(f"\nEER target:    {'✓ PASS' if eer_pass else '✗ FAIL'}")
print(f"Cosine target: {'✓ PASS' if cos_pass else '✗ FAIL'}")

# Save results
results = {
    "model": CONFIG.model.name,
    "best_epoch": best_ckpt["epoch"],
    "test_eer": test_eer,
    "test_cosine_same_speaker": test_cosine,
    "training_time_hours": round(total_time / 3600, 2),
}
with open(CKPT_DIR / "training_results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {CKPT_DIR / 'training_results.json'}")

Loaded best model from epoch 1
  test: 180 clips, 3 speakers

FINAL TEST RESULTS
EER:            0.2006 (target: ≤ 0.12)
Cosine (same):  0.5774 (target: ≥ 0.7)

EER target:    ✗ FAIL
Cosine target: ✗ FAIL

Results saved to /home/nibiru/Documents/sem6project/Speech2/pipeline_v2/checkpoints/speaker_encoder/training_results.json


---
## 4.10 — Inference Speed Test

In [13]:
# Measure inference latency (CPU)
encoder.mods.eval()
test_audio = torch.randn(1, 5 * 16000)  # 5 seconds of audio

# Warmup
for _ in range(3):
    _ = extract_embedding(encoder, test_audio)

# Time 10 runs
times = []
for _ in range(10):
    t0 = time.time()
    emb = extract_embedding(encoder, test_audio)
    times.append(time.time() - t0)

avg_time = np.mean(times)
print(f"Inference time (5s audio → 192-dim embedding):")
print(f"  Mean:    {avg_time*1000:.1f} ms")
print(f"  Target:  ≤ {CONFIG.target_metrics.inference_time_sec * 1000:.0f} ms")
print(f"  Status:  {'✓ PASS' if avg_time <= CONFIG.target_metrics.inference_time_sec else '✗ FAIL'}")

del test_audio, emb
gc.collect()

Inference time (5s audio → 192-dim embedding):
  Mean:    241.7 ms
  Target:  ≤ 500 ms
  Status:  ✓ PASS


1138

---
## ✓ Notebook 04 Complete

**What we accomplished:**
- Fine-tuned ECAPA-TDNN on RAVDESS speaker data
- AAM-Softmax loss with angular margin (m=0.2, s=30)
- **NEW: Gradient Reversal Layer (GRL)** for language-agnostic embeddings
  - Adversarial language classifier (English vs Telugu)
  - FLEURS Telugu data for cross-lingual GRL training
  - λ annealing: 0 → 1 over first half of training
- Contrastive batch sampling (4 speakers × 4 utterances)
- EER evaluation with cosine similarity
- Inference speed test

**Why GRL matters for voice preservation:**
Without GRL, the speaker encoder learns embeddings that encode both speaker identity AND language characteristics. During zero-shot Telugu→English translation, this causes the TTS to produce output that sounds like a different speaker because the "Telugu accent" encoded in the embedding clashes with English synthesis. GRL forces the encoder to produce embeddings that are **language-invariant**, preserving only the speaker's voice timbre.

**Target:** EER ≤ 12%, cosine(same-speaker) ≥ 0.70  
**Next:** Open `05_translation_finetuning.ipynb`